Naložimo knjižnico SageMath.
> Ubuntu + SageMath ~ 15GB prostora

> Docker Desktop (Ubuntu) + SageMath ~ 8GB prostora

In [19]:
from sage.all import *

Definiramo metriko razdalja, ki določi razdaljo med vozlišči v grafu. Razdalja = 
- 0 ... ko primerjamo vozlišče samega s seboj
- 1 ... ko primerjamo sosednji vozlišči
- 2 ... ko primerjamo nesosednji vozlišči

Naj bo:
- n ... število vozlišč,
- d ... število generatorjev.

Predpostavimo:
- n > 2d

In [20]:
def razdalja(w, u, n, d):
    if n <= 2 * d:
        raise ValueError("Pogoj n > 2d mora biti izpolnjen.")
    if w == u:
        return 0
    korak = (u - w) % n # (u-w)mod n ... ostanek pri deljenju z n
    # korak = i za i iz {1,..., n-d} predstavlja povezavo med 0. in i. vozliščem 
    return 1 if 1 <= korak <= n - d else 2  # za korake > n-d ni direktne povezave

Koda od Maše ___ . Prilagojena našemu problemu. Določamo metrično razdaljo, torej razdaljo med točko in množico točk.

In [21]:

def metricna_dimenzija_Cn(n, d, prelomi_simetrijo=True):
    """
    Izračuna metrično dimenzijo usmerjenega cirkulantnega grafa C_n(1, ..., n-d).
    Predpostavka: n > 2*d (zato so razdalje le 0, 1 ali 2).
    """
    if n <= 2 * d:
        raise ValueError("Pogoj n > 2d mora biti izpolnjen.")
    
    # Nastavitev linearnega programa (ILP)
    lp = MixedIntegerLinearProgram(maximization=False)
    x = lp.new_variable(binary=True) # izbere novo vozlišče iz grafa
    
    # Ciljna funkcija: minimiziramo število izbranih radarjev
    lp.set_objective(sum(x[w] for w in range(n)))
    
    # Lomljenje rotacijske simetrije (priporočeno v navodilih za hitrost pri večjih n)
    if prelomi_simetrijo:
        lp.add_constraint(x[0] == 1)

    # Pogoji: vsak par (u, v) mora biti ločen z vsaj enim radarjem
    for u in range(n):
        for v in range(u + 1, n):  # Gledamo samo u < v (brez podvajanja!)
            # Poiščemo vsa vozlišča w, ki ločijo par u in v
            locevalci = [w for w in range(n) if razdalja(w, u, n, d) != razdalja(w, v, n, d)]
            
            # Dodamo pogoj: vsaj eden izmed ločevalcev mora biti izbran
            lp.add_constraint(sum(x[w] for w in locevalci) >= 1)
            
    # Rešitev modela
    lp.solve()
    
    # Pridobimo izbrana vozlišča (pazimo na zaokroževanje pri float vrednostih)
    razresujoca_mnozica = [w for w in range(n) if round(lp.get_values(x[w])) == 1]
    dimenzija = len(razresujoca_mnozica)
    
    return dimenzija, razresujoca_mnozica

In [22]:
def je_razresujoca_mnozica(n, d, W):
    """Preverimo če je množica W razrešujoča. Torej da so vsi vektorji paroma različni."""
    if len(W) != len(set(W)) or any(w < 0 or w >= n for w in W):
        return False
    podpisi = [tuple(razdalja(w, v, n, d) for w in W) for v in range(n)]
    return len(set(podpisi)) == n

In [23]:
def test_znanih_formul(n_max=40):
    """Testiramo ILP za že poznane rezultate pri d=4 in d=5."""
    if n_max < 11:
        raise ValueError("n_max mora biti večji od 8 in 10.")

    stevilo_testov = 0
    for d, n_min in ((4, 10), (5, 11)):
        for n in range(n_min, n_max + 1):
            dimenzija, W = metricna_dimenzija_Cn(n, d)

            if len(W) != dimenzija:
                raise AssertionError(f"Moč množice in dmenzija sta različni pri n={n}, d={d}.")
            if not je_razresujoca_mnozica(n, d, W):
                raise AssertionError(f"Vrnjena množica ni razrešujoča za n={n}, d={d}.")

            if d == 4:
                expected = (2 * n + 4) // 5
            elif n % 3 == 0:
                expected = n // 3
            else:
                expected = n // 3 + 2

            if dimenzija != expected:
                raise AssertionError(
                    f"Pri n={n}, d={d}: dobimo {dimenzija}, pričakujemo pa {expected}."
                )
            stevilo_testov += 1

    print(f"Vseh {stevilo_testov} testov je bilo pravilnih.")

In [24]:
test_znanih_formul(n_max=40)

Vseh 61 testov je bilo pravilnih.


## Optimiziran solver za dolgi eksperiment

Ta razlicica uporabi HiGHS, Sageovo hitro metodo `lp.sum` (zaradi nižje časovne zahtevnosti napram navadnem sum) in rotacijsko simetrijo tudi pri pripravi omejitev. Ohrani isti vmesnik `metricna_dimenzija_Cn`, zato preostala koda ostane zdruzljiva.

In [25]:
def pripravi_model_Cn(
    n, d, prelomi_simetrijo=True, solver="HiGHS",
    casovna_omejitev=None, prikazi_dnevnik=False):
    """Pripravi ILP za C_n(1, ..., n-d), vendar ga se ne resi."""

    if n <= 2 * d:
        raise ValueError("Pogoj n > 2d mora biti izpolnjen.")

    lp = MixedIntegerLinearProgram(maximization=False, solver=solver)
    x = lp.new_variable(binary=True)

    # lp.sum je v Sage bistveno hitrejsi od Pythonovega sum za linearne izraze.
    lp.set_objective(lp.sum(x[w] for w in range(n)))

    if prelomi_simetrijo:
        lp.add_constraint(x[0] == 1)

    # Zaradi rotacijske simetrije locila za (u,v) dobimo z rotacijo locil
    # za (0,v-u). Razdalje zato izracunamo O(n^2)-krat namesto O(n^3)-krat.
    locila_po_razliki = {}
    for delta in range(1, n):
        locila_po_razliki[delta] = tuple(
            s for s in range(n)
            if razdalja(s, 0, n, d) != razdalja(s, delta, n, d))

    for u in range(n):
        for v in range(u + 1, n):
            delta = v - u
            lp.add_constraint(
                lp.sum(x[(u + s) % n] for s in locila_po_razliki[delta]) >= 1)

    if casovna_omejitev is not None:
        if str(solver).casefold() == "highs":
            lp.solver_parameter("time_limit", float(casovna_omejitev))
        else:
            lp.solver_parameter("timelimit", float(casovna_omejitev))

    if str(solver).casefold() == "highs":
        # Nicelna vrzel: sprejmemo samo dokazani optimum.
        lp.solver_parameter("mip_rel_gap", 0.0)
        lp.solver_parameter("mip_abs_gap", 0.0)
        lp.solver_parameter("log_to_console", bool(prikazi_dnevnik))

    return lp, x


def resi_pripravljeni_model(lp, x, n):
    """Resi pripravljen model in vrne dimenzijo ter optimalno mnozico."""
    lp.solve()
    vrednosti = lp.get_values(x, convert=ZZ, tolerance=1e-6)
    W = [w for w in range(n) if vrednosti[w] == 1]
    return len(W), W


def metricna_dimenzija_Cn(
    n, d, prelomi_simetrijo=True, solver="HiGHS",
    casovna_omejitev=None, prikazi_dnevnik=False):
    """Izracuna tocno metricno dimenzijo in optimalno razresujoco mnozico."""

    lp, x = pripravi_model_Cn(
        n, d, prelomi_simetrijo=prelomi_simetrijo, solver=solver,
        casovna_omejitev=casovna_omejitev, prikazi_dnevnik=prikazi_dnevnik)
    return resi_pripravljeni_model(lp, x, n)

## Eksperiment za $4 \leq d \leq 15$

Rezultat vsake instance se takoj doda v CSV, zato lahko prekinjen poskus pozneje nadaljujemo. Primeri so urejeni po narascajocem $n$, da pred tezjimi primeri dobimo cim vec rezultatov za vse vrednosti $d$. Casovna omejitev velja za vsak ILP posebej.

In [26]:
import csv
import json
from datetime import datetime
from pathlib import Path
from time import perf_counter
from sage.numerical.mip import MIPSolverException

STOLPCI_REZULTATOV = [
    "timestamp", "n", "d", "dimension", "resolving_set",
    "status", "certificate_ok", "runtime_seconds",
    "build_seconds", "solve_seconds", "solver",
    "timeout_seconds", "symmetry_break", "error"]


def _zadnji_rezultati(pot):
    """Vrne zadnjo zapisano vrstico za vsak par (n,d)."""
    pot = Path(pot)
    zadnji = {}
    if not pot.exists():
        return zadnji
    with pot.open("r", newline="", encoding="utf-8") as datoteka:
        for vrstica in csv.DictReader(datoteka):
            zadnji[(int(vrstica["n"]), int(vrstica["d"]))] = vrstica
    return zadnji


def _dodaj_vrstico(pot, vrstica):
    """Doda en dokoncan rezultat v checkpoint CSV."""
    pot = Path(pot)
    pot.parent.mkdir(parents=True, exist_ok=True)
    zapisi_glavo = not pot.exists() or pot.stat().st_size == 0
    with pot.open("a", newline="", encoding="utf-8") as datoteka:
        pisec = csv.DictWriter(datoteka, fieldnames=STOLPCI_REZULTATOV)
        if zapisi_glavo:
            pisec.writeheader()
        pisec.writerow(vrstica)


def izracunaj_instanco(
    n, d, casovna_omejitev=300, solver="HiGHS",
    prelomi_simetrijo=True, prikazi_dnevnik=False):
    """Izracuna eno instanco in vrne vrstico za CSV."""

    zacetek = perf_counter()
    zacetek_resevanja = None
    vrstica = {
        "timestamp": datetime.now().astimezone().isoformat(timespec="seconds"),
        "n": int(n), "d": int(d), "dimension": "",
        "resolving_set": "", "status": "error",
        "certificate_ok": False, "runtime_seconds": "",
        "build_seconds": "", "solve_seconds": "",
        "solver": solver, "timeout_seconds": float(casovna_omejitev),
        "symmetry_break": bool(prelomi_simetrijo), "error": ""}

    try:
        t0 = perf_counter()
        lp, x = pripravi_model_Cn(
            n, d, prelomi_simetrijo=prelomi_simetrijo, solver=solver,
            casovna_omejitev=casovna_omejitev, prikazi_dnevnik=prikazi_dnevnik)
        vrstica["build_seconds"] = round(perf_counter() - t0, 6)

        t0 = perf_counter()
        zacetek_resevanja = t0
        dimenzija, W = resi_pripravljeni_model(lp, x, n)
        vrstica["solve_seconds"] = round(perf_counter() - t0, 6)

        certifikat = je_razresujoca_mnozica(n, d, W)
        vrstica.update({
            "dimension": int(dimenzija),
            "resolving_set": json.dumps([int(w) for w in W]),
            "status": "optimal" if certifikat else "invalid_certificate",
            "certificate_ok": bool(certifikat)})

    except MIPSolverException as napaka:
        sporocilo = str(napaka)
        vrstica["status"] = (
            "timeout" if "time limit" in sporocilo.casefold() else "solver_error")

        vrstica["error"] = sporocilo

    except KeyboardInterrupt:
        print("\nUporabnik je prekinil instanco; ta ni bila zapisana.", flush=True)
        raise

    except Exception as napaka:
        vrstica["status"] = "error"
        vrstica["error"] = f"{type(napaka).__name__}: {napaka}"
        
    finally:
        if zacetek_resevanja is not None and vrstica["solve_seconds"] == "":
            vrstica["solve_seconds"] = round(perf_counter() - zacetek_resevanja, 6)
        vrstica["runtime_seconds"] = round(perf_counter() - zacetek, 6)

    return vrstica

In [27]:
def izvedi_eksperiment(
    d_vrednosti=range(4, 16), n_max=100, cas_na_instanco=300,
    skupni_cas_ur=8, pot="results/metric_dimensions.csv",
    solver="HiGHS", prelomi_simetrijo=True, prikazi_dnevnik=False,
    ustavi_d_po_timeoutu=True, ponovi_statuse=()):
    """
    Resuje dopustne pare (n,d), sproti shranjuje in podpira nadaljevanje.

    Za ponovni poskus timeoutov uporabi ponovi_statuse=("timeout",).
    Nova vrstica se doda v CSV; pri analizi obdrzimo zadnjo za vsak (n,d).
    """

    d_vrednosti = sorted({int(d) for d in d_vrednosti})
    if not d_vrednosti or min(d_vrednosti) < 4 or max(d_vrednosti) > 15:
        raise ValueError("d_vrednosti morajo biti med 4 in 15.")
    if n_max <= 2 * min(d_vrednosti):
        raise ValueError("n_max je premajhen za izbrane vrednosti d.")
    if cas_na_instanco <= 0 or skupni_cas_ur <= 0:
        raise ValueError("Casovni omejitvi morata biti pozitivni.")

    # Po n: najprej resimo lahke primere za vse d, sele nato vecje.
    kandidati = sorted(
        (n, d)
        for d in d_vrednosti
        for n in range(2 * d + 1, int(n_max) + 1))

    zadnji = _zadnji_rezultati(pot)
    ponovi_statuse = set(ponovi_statuse)
    prejsnji_timeout_d = {
        d for (n, d), vrstica in zadnji.items()
        if vrstica["status"] == "timeout" and "timeout" not in ponovi_statuse}

    kandidati = [ par for par in kandidati
        if par not in zadnji or zadnji[par]["status"] in ponovi_statuse]

    zacetek_eksperimenta = perf_counter()
    skupna_omejitev = 3600.0 * float(skupni_cas_ur)
    blokirani_d = set(prejsnji_timeout_d) if ustavi_d_po_timeoutu else set()
    uspesni = timeouti = 0

    print(f"Nacrt: {len(kandidati)} se neizracunanih primerov, "
        f"d={d_vrednosti}, n<={n_max}, solver={solver}.", flush=True)
    print(f"Omejitev: {cas_na_instanco:.0f} s/primer, "
        f"{skupni_cas_ur:g} h skupaj. CSV: {Path(pot)}", flush=True)

    if blokirani_d:
        print(f"Zaradi prejsnjih timeoutov preskakujem d={sorted(blokirani_d)}.", flush=True)

    for zaporedna, (n, d) in enumerate(kandidati, start=1):
        if d in blokirani_d:
            continue

        preteklo = perf_counter() - zacetek_eksperimenta
        preostalo = skupna_omejitev - preteklo

        if preostalo < 1.0:
            print("STOP: dosezena je skupna casovna omejitev.", flush=True)
            break

        trenutni_timeout = min(float(cas_na_instanco), preostalo)
        ura = datetime.now().astimezone().strftime("%H:%M:%S")
        print(f"\n[{zaporedna}/{len(kandidati)} | {ura}] START d={d}, n={n}; "
            f"solver najvec {trenutni_timeout:.0f} s.", flush=True)

        vrstica = izracunaj_instanco(
            n, d, casovna_omejitev=trenutni_timeout, solver=solver,
            prelomi_simetrijo=prelomi_simetrijo, prikazi_dnevnik=prikazi_dnevnik)

        _dodaj_vrstico(pot, vrstica)

        status = vrstica["status"]
        cas = float(vrstica["runtime_seconds"])
        if status == "optimal":
            uspesni += 1
            print(f"KONCANO d={d}, n={n}: dim={vrstica['dimension']}, "
                f"cas={cas:.2f} s, certifikat=OK.", flush=True)
            
        elif status == "timeout":
            timeouti += 1
            print(f"TIMEOUT d={d}, n={n} po {cas:.2f} s.", flush=True)

            if ustavi_d_po_timeoutu:
                blokirani_d.add(d)
                print(f"Nadaljnji n za d={d} bodo v tem zagonu preskoceni.", flush=True)

        else:
            print(f"STOP zaradi statusa {status}: {vrstica['error']}", flush=True)
            break

    skupaj = perf_counter() - zacetek_eksperimenta
    print(
        f"\nEksperiment koncan: {uspesni} optimalnih, {timeouti} timeoutov, "
        f"{skupaj / 60:.1f} min. Rezultati: {Path(pot)}", flush=True)
    return preberi_rezultate(pot)


def preberi_rezultate(pot="results/metric_dimensions.csv"):
    """Prebere CSV brez dodatnih knjiznic in obdrzi zadnji (n,d)."""
    zadnji = _zadnji_rezultati(pot)
    tabela = []
    for vrstica in zadnji.values():
        vrstica = dict(vrstica)
        vrstica["n"] = int(vrstica["n"])
        vrstica["d"] = int(vrstica["d"])
        vrstica["dimension"] = (
            int(float(vrstica["dimension"])) if vrstica["dimension"] else None)
        for ime in ("runtime_seconds", "build_seconds", "solve_seconds"):
            vrstica[ime] = float(vrstica[ime]) if vrstica[ime] else None
        tabela.append(vrstica)
    return sorted(tabela, key=lambda r: (r["d"], r["n"]))


def prikazi_povzetek(tabela, sirina_bloka=15):
    """Prikaze zadnje rezultate ter matriki dimenzij in casov."""
    from collections import Counter
    from html import escape
    from IPython.display import HTML, display

    def prikazi_html(vrstice):
        """Prikaze navadne HTML celice brez Sage/LaTeX ovojev \\( ... \\)."""
        glava, *podatki = vrstice
        deli = [
            '<div style="overflow-x:auto"><table style="border-collapse:collapse">',
            '<thead><tr>',
        ]
        deli.extend(
            f'<th style="border:1px solid #888;padding:4px 8px">{escape(str(x))}</th>'
            for x in glava
        )
        deli.append('</tr></thead><tbody>')
        for vrstica in podatki:
            deli.append('<tr>')
            deli.extend(
                f'<td style="border:1px solid #888;padding:4px 8px">'
                f'{escape("" if x is None else str(x))}</td>'
                for x in vrstica
            )
            deli.append('</tr>')
        deli.append('</tbody></table></div>')
        display(HTML(''.join(deli)))

    if not tabela:
        print("Tabela rezultatov je prazna.")
        return None, None

    stolpci = ["n", "d", "dimension", "status",
               "runtime_seconds", "build_seconds", "solve_seconds"]
    zadnjih_20 = [[r.get(ime, "") for ime in stolpci] for r in tabela[-20:]]
    print("Zadnjih 20 rezultatov:")
    prikazi_html([stolpci] + zadnjih_20)

    stevci = Counter(r["status"] for r in tabela)
    print("\nStevilo rezultatov po statusu:")
    prikazi_html([["status", "stevilo"]] + sorted(stevci.items()))

    optimalni = [r for r in tabela if r["status"] == "optimal"]
    if not optimalni:
        print("Ni optimalnih rezultatov za matriki.")
        return None, None

    matrika_dimenzij = {(r["d"], r["n"]): r["dimension"] for r in optimalni}
    matrika_casov = {(r["d"], r["n"]): r["runtime_seconds"] for r in optimalni}
    d_ji = sorted({r["d"] for r in optimalni})
    n_ji = sorted({r["n"] for r in optimalni})

    def izpisi_matriko(vrednosti, naslov, zaokrozi=False):
        print(f"\n{naslov}")
        for zacetek in range(0, len(n_ji), sirina_bloka):
            stolpci_n = n_ji[zacetek:zacetek + sirina_bloka]
            vrstice = [["d \\ n"] + stolpci_n]
            for d in d_ji:
                vrednosti_d = []
                for n in stolpci_n:
                    vrednost = vrednosti.get((d, n), "")
                    if zaokrozi and vrednost != "":
                        vrednost = round(vrednost, 3)
                    vrednosti_d.append(vrednost)
                vrstice.append([d] + vrednosti_d)
            prikazi_html(vrstice)

    izpisi_matriko(matrika_dimenzij, "Matrika metricnih dimenzij:")
    izpisi_matriko(matrika_casov, "Matrika celotnih casov v sekundah:", True)
    return matrika_dimenzij, matrika_casov

### Kratek preizkus (neobvezno)

Ta celica resi samo tri majhne primere in pise v loceno datoteko. Uporabi jo pred dolgim zagonom.

In [28]:
testna_tabela = izvedi_eksperiment(
    d_vrednosti=[4], n_max=11, cas_na_instanco=30, skupni_cas_ur=0.05,
    pot="results/test_metric_dimensions.csv", prikazi_dnevnik=False)
    
prikazi_povzetek(testna_tabela)

Nacrt: 0 se neizracunanih primerov, d=[4], n<=11, solver=HiGHS.
Omejitev: 30 s/primer, 0.0500000000000000 h skupaj. CSV: results/test_metric_dimensions.csv

Eksperiment koncan: 0 optimalnih, 0 timeoutov, 0.0 min. Rezultati: results/test_metric_dimensions.csv
Zadnjih 20 rezultatov:


n,d,dimension,status,runtime_seconds,build_seconds,solve_seconds
9,4,4,optimal,0.20694,0.096468,0.10659
10,4,4,optimal,0.047439,0.012148,0.034439
11,4,5,optimal,0.052463,0.020659,0.031502



Stevilo rezultatov po statusu:


status,stevilo
optimal,3



Matrika metricnih dimenzij:


d \ n,9,10,11
4,4,4,5



Matrika celotnih casov v sekundah:


d \ n,9,10,11
4,0.207,0.047,0.052


({(4, 9): 4, (4, 10): 4, (4, 11): 5},
 {(4, 9): 0.20694, (4, 10): 0.047439, (4, 11): 0.052463})

### Glavni zagon

Privzeto ima vsak primer najvec 5 minut, celoten zagon pa 8 ur. `START` vedno izpise tudi najdaljsi mozni cas trenutnega primera. Za podrobni HiGHS dnevnik nastavi `PRIKAZI_DNEVNIK = True`; privzeto je izklopljen, da izhod skoraj tisoc primerov ne upocasni VS Code. Ponoven zagon nadaljuje iz obstojecga CSV.

In [ ]:
D_VREDNOSTI = range(4, 16)
N_MAX = 100
CAS_NA_INSTANCO = 5 * 60       # 5 minut
SKUPNI_CAS_UR = 8              # spremeni po potrebi
PRIKAZI_DNEVNIK = False        # True le za podrobni HiGHS dnevnik
POT_REZULTATOV = "results/metric_dimensions.csv"

In [ ]:
rezultati = izvedi_eksperiment(
    d_vrednosti=D_VREDNOSTI, n_max=N_MAX,
    cas_na_instanco=CAS_NA_INSTANCO, skupni_cas_ur=SKUPNI_CAS_UR,
    pot=POT_REZULTATOV, prikazi_dnevnik=PRIKAZI_DNEVNIK)
    
matrika_dimenzij, matrika_casov = prikazi_povzetek(rezultati)